In [8]:
from __future__ import annotations

import earthaccess
import tqdm as notebook_tqdm
import json
from pathlib import Path

For documentation for eartchacess, see [nasa_website](https://earthaccess.readthedocs.io/en/stable/api/#earthaccess.api.search_datasets).

|Product|Type|
|---|---|
|FIRMS MCD14DL/MCD14ML|Individual active-fire detections as points|
|MOD14A1 and MYD14A1|Gridded fire mask|

In [9]:
def granule_record(granule: Any) -> dict[str, Any]:
    meta = granule.get("meta", {})
    umm = granule.get("umm", {})
    temporal = umm.get("TemporalExtent", {}).get("RangeDateTime", {})
    return {
        "concept_id": meta.get("concept-id"),
        "granule_ur": umm.get("GranuleUR"),
        "beginning_datetime": temporal.get("BeginningDateTime"),
        "ending_datetime": temporal.get("EndingDateTime"),
        "data_links": granule.data_links(access="external"),
    }

In [ ]:
DEFAULT_BBOX = (-82.0, -21.0, -49.0, 6.0)
SHORT_NAME = "MYD14A1"
VERSION = "061"
YEAR = 2023
ARG_DOWNLOAD = True

OUTPUT_DIR = Path(f'../../modis_{SHORT_NAME.lower()}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

download_dir = OUTPUT_DIR / 'download'
download_dir.mkdir(parents=True, exist_ok=True)
manifest_dir = OUTPUT_DIR / 'manifest' 
manifest_dir.mkdir(parents=True, exist_ok=True)

total = 0


# 2. Search
earthaccess.login(strategy="netrc", persist=True)
for day in range(1, 31+1):
    search_results = earthaccess.search_data(
        short_name= SHORT_NAME,  # ATLAS/ICESat-2 L3A Land Ice Height
        bounding_box=DEFAULT_BBOX,
        # (lower_left_lon, lower_left_lat, upper_right_lon, upper_right_lat)
        
        # Only include files in area of interest...
        temporal=(f"2023-01-{"{:02d}".format(day)}T00:00:00", 
                  f"2023-01-{"{:02d}".format(day)}T23:59:59"
        ),  # ...and time period of interest.
        count=10
    )
    records = [granule_record(search_result) for search_result in search_results]
    manifest = {"dataset": f"{SHORT_NAME}.{VERSION}", 
                "year": YEAR,
                "day": "{:02d}".format(day),
                "bbox": list(DEFAULT_BBOX), 
                "granule_count": len(records),
                "granules": records
    }
    manifest_path = OUTPUT_DIR / 'manifest' / f"manifest_{YEAR}_Jan_{"{:02d}".format(day)}.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    total += len(search_results)
    if search_results and ARG_DOWNLOAD:
        earthaccess.download(search_results, download_dir)

print(f"Total matched granules across this month (2023_Jan): {total}")



# search_results